# Algorithms and Processes of a No-Limit Texas Hold'em Engine

### Annotated Python notebook: theory, mathematics, algorithms, and verification

This notebook turns the supplied `no_limit_holdem_engine_algorithms.py` into a guided, executable reference. It follows the attached *Algorithms and Processes of a No-Limit Hold'em Engine* PDF and keeps the supplied implementation as the starting point.

The goal is to explain not only **what** each function does, but **why** the algorithm works, what mathematics it uses, what assumptions it makes, and which checks support its correctness.

> **How to use the notebook:** Run cells from top to bottom. Most code uses only the Python standard library. Random simulation cells can produce slightly different Monte Carlo estimates on each run; cells that use a seeded random generator are reproducible. Exact-enumeration results are deterministic.

**Source note.** The notebook documents the provided Python reference implementation. When the documentation's prose makes a stronger claim than the code actually implements, the difference is explicitly called out rather than silently assumed to be correct.

## Learning map

| Stage | What you will learn | Main mathematics / algorithms |
|---|---|---|
| 1. Foundations | The rules, combinations, EV, pot odds, MDF and bluff fractions | \(\binom{n}{k}\), expectation, break-even inequalities |
| 2. Cards and randomness | Compact card IDs and fair shuffling | Bit operations, rejection sampling, Fisher-Yates |
| 3. Hand evaluation | How five/seven-card hands are ranked | Histograms, lexicographic scores, bitmasks, lookup tables |
| 4. Equity | How likely a hand is to win a share of the pot | Conditional probability, Monte Carlo, standard error, exact enumeration |
| 5. Betting and pots | How turns, raises, all-ins and chips are managed | State transitions, conservation, contribution bands |
| 6. AI and adaptation | How the program chooses actions and models opponents | Heuristics, simplified CFR demonstration, Beta-Binomial updating |
| 7. Verification | What the example assertions do—and do not—prove | Invariants, differential tests, worked hand |

### A useful distinction

A **rules engine** decides what actions are legal and where chips go. A **hand evaluator** compares cards. An **equity calculator** estimates expected showdown pot share. A **strategy policy** chooses actions. These layers interact, but success in one layer does not prove the others are correct.

## 1. Foundations: poker, probability, and decisions

### 1.1 The hand as a sequence of public and private information

Texas Hold'em uses a 52-card deck. Each player receives two private *hole cards*. Up to five community cards are revealed as the flop (three), turn (one), and river (one). At showdown, each remaining player uses the best five-card poker hand available from their two hole cards plus the five-card board.

A player may use both hole cards, one, or neither. The board itself can be the best five-card hand for every player, resulting in a split pot.

### 1.2 Counting outcomes with combinations

The number of ways to select \(k\) distinct objects from \(n\) when order does not matter is

\[
\binom{n}{k}=\frac{n!}{k!(n-k)!}.
\]

For example, there are

\[
\binom{52}{2}=\frac{52\cdot51}{2}=1,326
\]

physical two-card starting combinations, and

\[
\binom{52}{5}=2,598,960
\]

distinct five-card hands. Seven available cards contain \(\binom{7}{5}=21\) different five-card subsets. This is why the readable evaluator can simply score all 21 possibilities and keep the best one.

**Hand-category convention:** the display/combinatorics table separates Royal Flush from the other Straight Flushes, giving ten display categories. The engine's packed evaluator uses nine numeric categories (High Card through Straight Flush) and treats a Royal Flush as an Ace-high Straight Flush. The ordering is the same.

### 1.3 Expected value and pot odds

Expected value (EV) is the probability-weighted average of the possible payoffs. It helps compare decisions over many repetitions; it does not predict the result of one hand.

Let \(P\) be the pot **after the opponent's bet but before our call**, let \(C\) be the additional call amount, and let \(q\) be our expected share of the final pot. Under the simplified assumption that no more chips go in after the call,

\[
EV(\text{call})=q(P+C)-C.
\]

Setting EV to zero gives the break-even threshold:

\[
q^*=\frac{C}{P+C}.
\]

**Example.** The pot is 150 after an opponent bets, and calling costs 50. The final pot after we call is 200, so \(q^*=50/200=25\%\). A call with equity above 25% has non-negative chip EV under this simplified model. Real decisions may require accounting for future betting, rake, tournament payout effects, and equity-estimation uncertainty.

### 1.4 MDF, bluff fractions, and SPR

If a bettor bets \(B\) into a pot \(P_0\), a pure bluff risks \(B\) to win \(P_0\). The break-even fold frequency is \(B/(P_0+B)\). Its complement gives the *minimum defense frequency* (MDF):

\[
MDF=\frac{P_0}{P_0+B}.
\]

For a simplified polarized river betting range containing only value hands and bluffs, the balanced bluff fraction is

\[
\alpha^*_{\text{bluff}}=\frac{B}{P_0+2B}.
\]

These are game-theoretic reference formulas under specific assumptions—not universal instructions to defend exactly MDF in every real hand. The **stack-to-pot ratio** (SPR) is effective remaining stack divided by the flop pot; low SPR means fewer chips remain relative to the pot.

### Code: foundational equations and helper functions

The following functions translate the formulas into calculations. The helper names make the units explicit: pot and cost should use the same chip units.

In [ ]:
import math
import os
import secrets
import time
from dataclasses import dataclass, field
from itertools import combinations
from typing import Dict, List, Optional, Tuple

def expected_value_call(equity: float, pot_before_call: float, call_cost: float) -> float:
    "Compute EV(Call) = q * (P + C) - C."
    return equity * (pot_before_call + call_cost) - call_cost

def pot_odds_threshold(pot_before_call: float, call_cost: float) -> float:
    "Break-even equity threshold q* = C / (P + C)."
    if call_cost <= 0:
        return 0.0
    return call_cost / (pot_before_call + call_cost)

def minimum_defense_frequency(pot_before_bet: float, bet_size: float) -> float:
    "MDF = P_0 / (P_0 + B) so an opponent's pure bluff has EV <= 0."
    return pot_before_bet / (pot_before_bet + bet_size)

def optimal_bluff_fraction(pot_before_bet: float, bet_size: float) -> float:
    "Optimal bluff share B / (P_0 + 2B) inside a polarized river bet."
    return bet_size / (pot_before_bet + 2.0 * bet_size)

def stack_to_pot_ratio(effective_stack: float, flop_pot: float) -> float:
    "SPR = S_eff / P_flop."
    return effective_stack / max(1.0, flop_pot)

### Code: exact five-card combinatorics and a bet-size reference table

Each category count below is derived from choices of ranks and suits. The assertion verifies that the ten display categories partition all 2,598,960 five-card hands without overlap or omission.

In [ ]:
# Verify combinatorial counts of all 10 five-card hand categories
comb = math.comb
CATEGORY_COMBINATORICS = [
    (9, "Royal Flush",     comb(4, 1)),
    (8, "Straight Flush",  9 * comb(4, 1)),
    (7, "Four of a Kind",  comb(13, 1) * comb(4, 4) * comb(12, 1) * comb(4, 1)),
    (6, "Full House",      comb(13, 1) * comb(4, 3) * comb(12, 1) * comb(4, 2)),
    (5, "Flush",           comb(4, 1) * comb(13, 5) - 40),
    (4, "Straight",        10 * (4**5 - 4)),
    (3, "Three of a Kind", comb(13, 1) * comb(4, 3) * comb(12, 2) * (comb(4, 1)**2)),
    (2, "Two Pair",        comb(13, 2) * (comb(4, 2)**2) * comb(11, 1) * comb(4, 1)),
    (1, "One Pair",        comb(13, 1) * comb(4, 2) * comb(12, 3) * (comb(4, 1)**3)),
    (0, "High Card",       (comb(13, 5) - 10) * (4**5 - 4)),
]

total_5card = comb(52, 5)
assert sum(cnt for _, _, cnt in CATEGORY_COMBINATORICS) == total_5card

print(f"{'Cat':>3} | {'Hand Category':<16} | {'Exact Count':>11} | {'Probability':>11}")
print("-" * 50)
for cat_id, name, count in CATEGORY_COMBINATORICS:
    print(f"{cat_id:>3} | {name:<16} | {count:>11,d} | {100.0 * count / total_5card:>10.6f}%")
print("-" * 50)
print(f"{'All':>3} | {'Total C(52,5)':<16} | {total_5card:>11,d} | {100.0:>10.6f}%\n")

print("Game-Theoretic Bet-Sizing Reference Table (Pot P_0 = 100):")
print(f"{'Bet Size':<14} | {'Pot Odds q*':>12} | {'MDF':>10} | {'Optimal Bluff %':>15}")
print("-" * 60)
for label, b in [("33% Pot (33)", 33.0), ("50% Pot (50)", 50.0), ("75% Pot (75)", 75.0), ("100% Pot (100)", 100.0), ("150% Overbet", 150.0)]:
    q_star = pot_odds_threshold(100.0 + b, b)
    mdf = minimum_defense_frequency(100.0, b)
    bluff_pct = optimal_bluff_fraction(100.0, b)
    print(f"{label:<14} | {100*q_star:>11.2f}% | {100*mdf:>9.2f}% | {100*bluff_pct:>14.2f}%")

Cat | Hand Category    | Exact Count | Probability
--------------------------------------------------
  9 | Royal Flush      |           4 |   0.000154%
  8 | Straight Flush   |          36 |   0.001385%
  7 | Four of a Kind   |         624 |   0.024010%
  6 | Full House       |       3,744 |   0.144058%
  5 | Flush            |       5,108 |   0.196540%
  4 | Straight         |      10,200 |   0.392465%
  3 | Three of a Kind  |      54,912 |   2.112845%
  2 | Two Pair         |     123,552 |   4.753902%
  1 | One Pair         |   1,098,240 |  42.256903%
  0 | High Card        |   1,302,540 |  50.117739%
--------------------------------------------------
All | Total C(52,5)    |   2,598,960 | 100.000000%

Game-Theoretic Bet-Sizing Reference Table (Pot P_0 = 100):
Bet Size       |  Pot Odds q* |        MDF | Optimal Bluff %
------------------------------------------------------------
33% Pot (33)   |       19.88% |     75.19% |          19.88%
50% Pot (50)   |       25.00% |     66.67% 

## 2. Process I — compact card representation and fair shuffling

### 2.1 Turning a card into an integer

Rather than store a card as an object such as `{"rank": "A", "suit": "spades"}`, the engine stores it as one integer from 0 to 51:

\[
c=4\rho+\sigma,
\]

where \(\rho\in\{0,\ldots,12\}\) is the rank index (2 through Ace) and \(\sigma\in\{0,1,2,3\}\) is the suit index.

Because \(4=2^2\), the rank can be recovered with `c >> 2` (shift off the two suit bits), and the suit with `c & 3` (keep the two lowest bits). For example, an Ace of Hearts has rank index 12 and suit index 1, so its card ID is \(4(12)+1=49\).

A compact representation simplifies indexing and makes bitmasks possible later.

### Code: card IDs, parsing, and display

The same numeric card can be displayed as `A♠` or parsed from shorthand such as `As`. The parser is a convenience layer; the evaluator works with integers.

In [ ]:
RANK_CHARS = "23456789TJQKA"
RANK_NAMES = [
    "Two", "Three", "Four", "Five", "Six", "Seven", "Eight",
    "Nine", "Ten", "Jack", "Queen", "King", "Ace"
]
RANK_PLURAL = [
    "Twos", "Threes", "Fours", "Fives", "Sixes", "Sevens", "Eights",
    "Nines", "Tens", "Jacks", "Queens", "Kings", "Aces"
]
SUIT_CHARS = "shdc"
SUIT_SYMBOLS = "♠♥♦♣"
FULL_DECK = tuple(range(52))

def encode_card(rank: int, suit: int) -> int:
    "Bijective encoding c = 4 * rank + suit."
    return (rank << 2) | suit

def card_rank(c: int) -> int:
    "Extract rank index rho(c) = c >> 2 in {0..12}."
    return c >> 2

def card_suit(c: int) -> int:
    "Extract suit index sigma(c) = c & 3 in {0..3}."
    return c & 3

def parse_card(token: str) -> int:
    "Parse shorthand like 'As', 'Kh', 'Td', '5c' into integer 0..51."
    token = token.strip()
    r = RANK_CHARS.index(token[0].upper())
    s_char = token[1].lower()
    s = SUIT_CHARS.index(s_char) if s_char in SUIT_CHARS else SUIT_SYMBOLS.index(token[1])
    return encode_card(r, s)

def parse_cards(tokens: str) -> List[int]:
    "Parse space-separated card strings like 'As Kh Qd Jc Ts'."
    return [parse_card(t) for t in tokens.split() if t.strip()]

def card_str(c: int) -> str:
    "Format integer card 0..51 as e.g. 'A♠'."
    return f"{RANK_CHARS[c >> 2]}{SUIT_SYMBOLS[c & 3]}"

def hand_str(cards: List[int]) -> str:
    return " ".join(card_str(c) for c in cards)

### 2.2 Why shuffle with Fisher-Yates?

A correct shuffle should give every one of the \(52!\) deck orders the same probability. Fisher-Yates constructs a permutation one position at a time: at index \(i\), choose one of the \(i+1\) still-unplaced cards uniformly and swap it into position \(i\).

To obtain a uniform integer in \(\{0,\ldots,n-1\}\) from a random 32-bit word, naive `x % n` can be biased if \(n\) does not divide \(2^{32}\). The code uses rejection sampling:

\[
L=\left\lfloor\frac{2^{32}}n\right\rfloor n.
\]

Draw \(x\) uniformly from \([0,2^{32})\); reject it if \(x\ge L\). The accepted interval contains an equal number of values for every remainder modulo \(n\), so `x % n` is uniform. The method assumes the operating-system randomness source supplies uniformly distributed bytes.

### Algorithm 1: unbiased integer sampling and in-place deck shuffle

The assertions verify the encoding round-trip and that a shuffled deck remains a permutation of all 52 cards. They do not statistically prove randomness; the fairness argument comes from the algorithm and the quality of the underlying random source.

In [ ]:

# HIGHLIGHTED ALGORITHM 1: CryptographicShuffle
#   - secure_random_int(n): unbiased rejection sampling over [0, 2^32)
#   - shuffle_deck_in_place(deck): Fisher-Yates permutation over all 52! decks

def secure_random_int(n: int) -> int:
    "Algorithm 1a: Unbiased random integer in {0, ..., n-1} via rejection sampling."
    if n <= 1:
        return 0
    two32 = 1 << 32
    limit = (two32 // n) * n  # Largest multiple of n <= 2^32
    while True:
        # Draw 32-bit unsigned integer from OS cryptographic entropy source
        x = int.from_bytes( os.urandom(4), "little" )
        if x < limit:
            return x % n

def shuffle_deck_in_place(deck: List[int]) -> List[int]:
    "Algorithm 1b: In-place Fisher-Yates cryptographic shuffle."
    for i in range(len(deck) - 1, 0, -1):
        j = secure_random_int(i + 1)
        deck[i], deck[j] = deck[j], deck[i]
    return deck


# Verification of Bijective Encoding & Cryptographic Shuffle
assert all(encode_card(card_rank(c), card_suit(c)) == c for c in range(52))
sample_deck = list(FULL_DECK)
shuffle_deck_in_place(sample_deck)
assert len(sample_deck) == 52 and set(sample_deck) == set(FULL_DECK)

print("Bijective Card Encoding Verification: PASS (all 52 cards map 1-to-1)")
print("Sample Cryptographically Shuffled Deck (first 13 cards):", hand_str(sample_deck[:13]))

Bijective Card Encoding Verification: PASS (all 52 cards map 1-to-1)
Sample Cryptographically Shuffled Deck (first 13 cards): 4♠ J♥ J♦ A♥ 6♦ 3♣ 8♦ J♠ 6♠ 5♥ 9♣ T♣ K♣


## 3. Process II — hand evaluation

### 3.1 Encode a hand so integer comparison gives poker order

A hand's category is compared first; only if categories match are the relevant tie-break ranks compared in order. A *kicker* is an otherwise unused card rank that breaks a tie—for example, A-A-K-7-3 beats A-A-Q-7-3 because King is the higher first kicker.

The code packs the category and five tie-break digits into a base-13 integer:

\[
V(\kappa, k)=\kappa\,13^5 + \sum_{i=0}^{4} k_i\,13^{4-i}.
\]

Every rank digit is from 0 through 12, so base 13 prevents one rank digit from spilling into another. The category is the most significant digit group, so a higher category always outranks a lower category. Within a category, the tuple is compared lexicographically.

The code uses category IDs 0–8; Royal Flush is encoded as an Ace-high Straight Flush, category 8. Decoding reverses the operation using repeated division and remainders by 13.

### Algorithms 2 and 6: pack and unpack a score

In [ ]:
# HIGHLIGHTED ALGORITHM 2: Encode (Base-13 Lexicographical Score Packing)

def encode_score(category: int, tie_breakers: List[int]) -> int:
    "Algorithm 2 (Encode): Pack category in {0..8} and up to 5 kickers into base-13 integer."
    v = category
    for i in range(5):
        v = v * 13 + (tie_breakers[i] if i < len(tie_breakers) else 0)
    return v


# HIGHLIGHTED ALGORITHM 6: Decode (Base-13 Score Unpacking & Hand Naming)

def decode_score(v: int) -> Tuple[int, List[int]]:
    "Algorithm 6 (Decode): Unpack a base-13 integer score into (category, [k0, k1, k2, k3, k4])."
    ks = [0] * 5
    x = v
    for i in range(4, -1, -1):
        ks[i] = x % 13
        x //= 13
    return x, ks

def format_hand_name(v: int) -> str:
    "Convert a packed base-13 score into a descriptive English hand title."
    cat, k = decode_score(v)
    if cat == 8:
        return "Royal Flush" if k[0] == 12 else f"Straight Flush, {RANK_NAMES[k[0]]} high"
    if cat == 7:
        return f"Four of a Kind, {RANK_PLURAL[k[0]]}"
    if cat == 6:
        return f"Full House, {RANK_PLURAL[k[0]]} full of {RANK_PLURAL[k[1]]}"
    if cat == 5:
        return f"Flush, {RANK_NAMES[k[0]]} high"
    if cat == 4:
        return f"Straight, {RANK_NAMES[k[0]]} high"
    if cat == 3:
        return f"Three of a Kind, {RANK_PLURAL[k[0]]}"
    if cat == 2:
        return f"Two Pair, {RANK_PLURAL[k[0]]} and {RANK_PLURAL[k[1]]}"
    if cat == 1:
        return f"Pair of {RANK_PLURAL[k[0]]}"
    return f"High Card, {RANK_NAMES[k[0]]}"

### 3.2 Classify exactly five cards (`eval5`)

The reference evaluator first counts how often each rank appears (a 13-element histogram), checks whether all suits match, and builds a list of ranks grouped by multiplicity. It then checks categories from strongest to weakest:

1. Straight Flush
2. Four of a Kind
3. Full House
4. Flush
5. Straight
6. Three of a Kind
7. Two Pair
8. One Pair
9. High Card

The Ace-low straight (A-2-3-4-5, called the *wheel*) needs a special case. Rank IDs use Ace as 12 and deuce as 0, so the wheel is assigned a five-high straight score (rank index 3), not an Ace-high score.

The straight test also requires five **distinct** ranks. Without that condition, a repeated rank could accidentally pass a simple max-minus-min test.

### Algorithm 3: five-card classifier

In [ ]:
# HIGHLIGHTED ALGORITHM 3: Eval5 (Exact 5-Card Hand Classifier)

def eval5(cards5: List[int]) -> int:
    "Algorithm 3 (Eval5): Classify and score an exact 5-card hand."
    s0 = cards5[0] & 3
    flush = all((c & 3) == s0 for c in cards5)

    cnt = [0] * 13
    for c in cards5:
        cnt[c >> 2] += 1

    # Distinct ranks sorted strictly descending
    uniq = [r for r in range(12, -1, -1) if cnt[r] > 0]
    groups: Dict[int, List[int]] = {1: [], 2: [], 3: [], 4: []}
    for r in uniq:
        groups[cnt[r]].append(r)

    straight = False
    s_top = -1
    if len(uniq) == 5:
        if uniq[0] - uniq[4] == 4:
            straight = True
            s_top = uniq[0]
        elif uniq[0] == 12 and uniq[1] == 3 and uniq[4] == 0:  # Wheel A-2-3-4-5
            straight = True
            s_top = 3

    if straight and flush:
        return encode_score(8, [s_top])
    if groups[4]:
        return encode_score(7, [groups[4][0], groups[1][0]])
    if groups[3] and groups[2]:
        return encode_score(6, [groups[3][0], groups[2][0]])
    if flush:
        return encode_score(5, uniq)
    if straight:
        return encode_score(4, [s_top])
    if groups[3]:
        return encode_score(3, [groups[3][0], groups[1][0], groups[1][1]])
    if len(groups[2]) == 2:
        return encode_score(2, [groups[2][0], groups[2][1], groups[1][0]])
    if len(groups[2]) == 1:
        return encode_score(1, [groups[2][0], groups[1][0], groups[1][1], groups[1][2]])
    return encode_score(0, uniq)

### 3.3 Best five cards from seven (`best7`)

A Hold'em showdown has up to seven available cards: two private cards plus five shared cards. Any five of those seven may form the hand, so

\[
\binom{7}{5}=\frac{7!}{5!2!}=21.
\]

`best7` is deliberately transparent: it evaluates every five-card subset and returns the highest score along with the five cards that produced it. It is useful as a correctness oracle for faster evaluators.

In [ ]:
# HIGHLIGHTED ALGORITHM 4: Best7 (Exhaustive 21-Subset 7-Card Evaluator)

def best7(cards: List[int]) -> Tuple[int, Tuple[int, ...]]:
    "Algorithm 4 (Best7): Evaluate all C(n,5) subsets and return (best_score, best_5_cards)."
    best_score = -1
    best_subset: Tuple[int, ...] = ()
    for sub in combinations(cards, 5):
        sc = eval5(sub)
        if sc > best_score:
            best_score = sc
            best_subset = sub
    return best_score, best_subset



### 3.4 Fast seven-card evaluator (`rank7fast`)

The faster evaluator avoids the 21-subset loop. It accumulates:

- a 13-bit mask of all ranks present;
- one 13-bit rank mask per suit;
- a rank-count array for pairs, trips, and quads.

A bitmask uses bit \(r\) to say whether rank \(r\) is present. For example, OR-ing rank bits combines two sets, and testing a run of five adjacent set bits detects a straight. Precomputed lookup tables provide population counts, the highest straight in a mask, and the top five ranks in a mask.

The algorithm checks the strongest possible category first. If a suit has at least five cards, only that suit needs to be searched for a flush or straight flush. Otherwise rank multiplicities determine quads, full houses, trips, two pair, pairs, or high card.

**Memory note:** the accompanying PDF's compact-memory estimate corresponds to fixed-width typed arrays in a JavaScript implementation. This Python script builds ordinary Python lists, whose real memory footprint is larger; the algorithm is still logically the same.

### Algorithm 5: lookup tables and `rank7fast`

In [ ]:
# HIGHLIGHTED ALGORITHM 5: Rank7Fast (Single-Pass Bitmask + Lookup Evaluator)

POPC = [0] * 8192
STR8 = [-1] * 8192
TOP5T = [0] * 8192

for m in range(8192):
    POPC[m] = m.bit_count()
    for r in range(8, -1, -1):
        if (m & (31 << r)) == (31 << r):
            STR8[m] = r + 4
            break
    if STR8[m] < 0 and (m & 0x100F) == 0x100F:  # Wheel A-2-3-4-5 mask
        STR8[m] = 3
    top5 = []
    for r in range(12, -1, -1):
        if (m >> r) & 1:
            top5.append(r)
            if len(top5) == 5:
                break
    val = 0
    for i in range(5):
        val = val * 13 + (top5[i] if i < len(top5) else 0)
    TOP5T[m] = val

P13_5 = 371293
P13_4 = 28561
P13_3 = 2197
P13_2 = 169

def rank7fast(cards: List[int]) -> int:
    "Algorithm 5 (Rank7Fast): Single-pass bitmask and lookup-table 5/6/7-card evaluator."
    s0 = s1 = s2 = s3 = 0
    all_mask = 0
    cnt = [0] * 13
    for c in cards:
        r = c >> 2
        u = c & 3
        b = 1 << r
        all_mask |= b
        cnt[r] += 1
        if u == 0:
            s0 |= b
        elif u == 1:
            s1 |= b
        elif u == 2:
            s2 |= b
        else:
            s3 |= b

    # Check if any suit has >= 5 cards (at most one suit can in a <=7-card hand)
    fm = s0 if POPC[s0] >= 5 else (s1 if POPC[s1] >= 5 else (s2 if POPC[s2] >= 5 else (s3 if POPC[s3] >= 5 else 0)))
    if fm:
        sf = STR8[fm]
        if sf >= 0:
            return 8 * P13_5 + sf * P13_4
        return 5 * P13_5 + TOP5T[fm]

    q0 = t0 = t1 = p0 = p1 = p2 = -1
    for r in range(12, -1, -1):
        k = cnt[r]
        if k == 4:
            q0 = r
        elif k == 3:
            if t0 < 0:
                t0 = r
            else:
                t1 = r
        elif k == 2:
            if p0 < 0:
                p0 = r
            elif p1 < 0:
                p1 = r
            elif p2 < 0:
                p2 = r

    if q0 >= 0:
        kicker = next(r for r in range(12, -1, -1) if r != q0 and cnt[r] > 0)
        return 7 * P13_5 + q0 * P13_4 + kicker * P13_3

    if t0 >= 0 and (t1 >= 0 or p0 >= 0):
        pair_rank = max(t1, p0)
        return 6 * P13_5 + t0 * P13_4 + pair_rank * P13_3

    st = STR8[all_mask]
    if st >= 0:
        return 4 * P13_5 + st * P13_4

    if t0 >= 0:
        k0 = k1 = -1
        for r in range(12, -1, -1):
            if cnt[r] == 1:
                if k0 < 0:
                    k0 = r
                else:
                    k1 = r
                    break
        return 3 * P13_5 + t0 * P13_4 + k0 * P13_3 + k1 * P13_2

    if p1 >= 0:
        k0 = next(r for r in range(12, -1, -1) if r != p0 and r != p1 and cnt[r] > 0)
        return 2 * P13_5 + p0 * P13_4 + p1 * P13_3 + k0 * P13_2

    if p0 >= 0:
        k0 = k1 = k2 = -1
        for r in range(12, -1, -1):
            if cnt[r] == 1:
                if k0 < 0:
                    k0 = r
                elif k1 < 0:
                    k1 = r
                else:
                    k2 = r
                    break
        return P13_5 + p0 * P13_4 + k0 * P13_3 + k1 * P13_2 + k2 * 13

    return TOP5T[all_mask]

### 3.5 Related evaluator architectures

The script includes two additional demonstrations:

- **Prime-product identity:** assign each rank a distinct prime and multiply the five primes. Unique factorization means a rank multiset has a unique product. This identifies repeated-rank structure, but suit and straight handling still require additional logic for a complete evaluator.
- **Quinary/combinadic dynamic-programming hash:** represent a seven-card hand by its rank multiplicities (0 through 4 at each of 13 ranks) and map that bounded-count vector to an integer index. Perfect-hash evaluators use compact indexes to retrieve precomputed hand values.

These are architectural demonstrations, not a complete drop-in implementation of every external evaluator named in the documentation. The actual `rank7fast`/`best7` comparison remains the main executable equivalence check in this notebook.

### Associated evaluator structures and differential tests

The next cell builds the prime-product function and quinary table, compares `rank7fast` against `best7` on randomized seven-card hands, round-trips scores through the encoder/decoder, and checks canonical edge cases.

In [ ]:
# ASSOCIATED EVALUATOR ARCHITECTURES:
#   1. Cactus Kev Prime-Product Rank Multiset Identifier
#   2. Quinary Combinadic DP Hash (PHEvaluator / @pokertools)

RANK_PRIMES = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41)

def cactus_kev_prime_product(cards5: List[int]) -> int:
    '''
    Compute the unique Fundamental-Theorem-of-Arithmetic prime product
    prod_{i=1}^5 p_{r_i} identifying any 5-card rank multiset without sorting.
    '''
    prod = 1
    for c in cards5:
        prod *= RANK_PRIMES[c >> 2]
    return prod

#  ASSOCIATED EVALUATOR ARCHITECTURE: Quinary Combinadic DP Hash (PHEvaluator)

def build_quinary_dp_table() -> List[List[List[int]]]:
    '''
    Build the PHEvaluator / @pokertools combinadic DP table DP[l][r][k]
    mapping any 13-element base-5 rank histogram q in {0..4}^13 with sum(q)=7
    bijectively into {0, ..., 49204}.
    '''
    choose = [[0] * 8 for _ in range(14)]
    for n in range(14):
        choose[n][0] = 1
        for k in range(1, min(n, 7) + 1):
            choose[n][k] = choose[n - 1][k - 1] + choose[n - 1][k]

    # num_ways[rem_ranks][cards_left]: ways to place cards_left into rem_ranks with max 4 per rank
    ways = [[0] * 8 for _ in range(14)]
    ways[0][0] = 1
    for r in range(1, 14):
        for k in range(8):
            ways[r][k] = sum(ways[r - 1][k - m] for m in range(min(4, k) + 1))

    dp = [[[0] * 8 for _ in range(13)] for _ in range(5)]
    for r in range(13):
        rem_ranks = 12 - r
        for l in range(5):
            for k in range(l, 8):
                # Sum of ways for strictly smaller multipliers m < l at position r
                dp[l][r][k] = sum(ways[rem_ranks][k - m] for m in range(l))
    return dp

QUINARY_DP = build_quinary_dp_table()

def hash_quinary_7(q: List[int]) -> int:
    "Bijective combinadic hash h(q) in {0, ..., 49204} for sum(q) == 7."
    h = 0
    rem = 7
    for r in range(13):
        qr = q[r]
        if qr > 0:
            h += QUINARY_DP[qr][r][rem]
            rem -= qr
    return h


# Verification of Invariant I4: Differential test of Rank7Fast vs Best7 & Encode/Decode
rng_deck = list(FULL_DECK)
mismatches = 0
for _ in range(5000):
    shuffle_deck_in_place(rng_deck)
    hand7 = rng_deck[:7]
    s_fast = rank7fast(hand7)
    s_best, best5_cards = best7(hand7)
    if s_fast != s_best:
        mismatches += 1
    cat, ks = decode_score(s_fast)
    assert encode_score(cat, ks) == s_fast

print(f"Differential Test (Rank7Fast vs Best7 over 5,000 random 7-card hands): {mismatches} mismatches!")
print(f"Quinary Combinadic DP Verification: N_quinary(13, 7) max hash + 1 = {hash_quinary_7([4, 3] + [0]*11) + 1:,d} (exact: 49,205)\n")

# Showcase canonical edge-case hand comparisons
canonical_Matchups = [
    ("Royal Flush vs King-High SF", "As Ks Qs Js Ts 2d 3c", "Ks Qs Js Ts 9s Ah Kd"),
    ("Six-High Straight vs Wheel",  "6s 5d 4h 3c 2s Ah Kd", "As 2d 3h 4c 5s Kh Qd"),
    ("Double-Trips Full House",     "Ks Kh Kd Qs Qh Qd 2c", "Ks Kh Kd Js Jh Jd Ac"),
]
for title, h1_str, h2_str in canonical_Matchups:
    h1, h2 = parse_cards(h1_str), parse_cards(h2_str)
    v1, v2 = rank7fast(h1), rank7fast(h2)
    assert v1 > v2
    print(f"{title:<28}: {format_hand_name(v1)} ({v1:,d}) > {format_hand_name(v2)} ({v2:,d})")

Differential Test (Rank7Fast vs Best7 over 5,000 random 7-card hands): 0 mismatches!
Quinary Combinadic DP Verification: N_quinary(13, 7) max hash + 1 = 49,205 (exact: 49,205)

Royal Flush vs King-High SF : Royal Flush (3,313,076) > Straight Flush, King high (3,284,515)
Six-High Straight vs Wheel  : Straight, Six high (1,599,416) > Straight, Five high (1,570,855)
Double-Trips Full House     : Full House, Kings full of Queens (2,563,899) > Full House, Kings full of Jacks (2,561,702)


## 4. Process III — probability, equity, and runout calculations

### 4.1 Win probability is not the same as equity

A hand may win the whole pot, lose, or tie and receive a fraction. Let \(X\) be the fraction of the pot the player receives on one simulated showdown. Then equity is

\[
q=\mathbb{E}[X].
\]

For a win, \(X=1\); for a loss, \(X=0\); for a two-way tie, \(X=1/2\); for a three-way tie, \(X=1/3\), and so on. Averaging fractional shares makes the estimate compatible with pot-odds calculations.

Visible cards must be removed before sampling. Two hole cards and \(b\) board cards leave \(52-2-b\) unknown cards: 50 preflop, 47 on the flop, 46 on the turn, and 45 on the river.

### 4.2 Chen score: a quick starting-hand heuristic

The Chen-style score converts two hole cards into a single approximate preflop-strength number. It rewards high cards, pairs, suited cards, and connected ranks, and penalizes gaps. This is computationally cheap and useful for ordering or filtering candidate hands.

It is **not** an exact probability of winning. Real preflop equity depends on opponent holdings, number of opponents, position, and card removal. The code builds `CHEN_PAIR`, a lookup table so that a pair's score can be retrieved in constant time during range sampling.

In [ ]:
import random


# HIGHLIGHTED ALGORITHM 16: ChenScore (Closed-Form Preflop Valuation)

def chen_score(c1: int, c2: int) -> float:
    "Algorithm 16 (ChenScore): Bill Chen's starting-hand formula."
    r1, r2 = c1 >> 2, c2 >> 2
    s1, s2 = c1 & 3, c2 & 3
    hi, lo = max(r1, r2), min(r1, r2)
    base_map = {12: 10.0, 11: 8.0, 10: 7.0, 9: 6.0}
    score = base_map.get(hi, (hi + 2) / 2.0)

    if r1 == r2:
        return max(5.0, score * 2.0)
    if s1 == s2:
        score += 2.0
    gap = hi - lo - 1
    if gap == 1:
        score -= 1.0
    elif gap == 2:
        score -= 2.0
    elif gap == 3:
        score -= 4.0
    elif gap >= 4:
        score -= 5.0
    if gap <= 1 and hi < 10:
        score += 1.0
    return score

# Precompute 52x52 CHEN_PAIR matrix for O(1) range rejection sampling
CHEN_PAIR = [[0.0] * 52 for _ in range(52)]
for a in range(52):
    for b in range(a + 1, 52):
        sc = chen_score(a, b)
        CHEN_PAIR[a][b] = sc
        CHEN_PAIR[b][a] = sc

### 4.3 Monte Carlo equity (`equity`)

Exact enumeration quickly becomes expensive because we must choose unknown board cards and opponent hole cards. Monte Carlo estimates the expectation by simulating \(N\) random legal deals:

\[
\hat q=\frac1N\sum_{i=1}^{N}X_i.
\]

Each trial:
1. partially shuffles the unseen deck to sample only the cards needed;
2. fills in the missing board cards;
3. evaluates the hero and each opponent;
4. gives the trial a fractional pot share (0, 1, or a tie share);
5. adds the share and its square to running totals.

The implementation reuses lists/buffers to reduce allocations. It performs partial Fisher-Yates rather than shuffling every unseen card, because only a prefix is needed.

For sample values \(X_i\), a plug-in estimate of the standard error of the sample mean is

\[
SE(\hat q)\approx\sqrt{\frac{\overline{X^2}-\bar X^2}{N}}.
\]

The code uses this population-denominator plug-in estimate; it is a practical approximation, not the unbiased \(N-1\) sample-variance convention. Standard error falls roughly as \(1/\sqrt{N}\), so cutting it in half typically requires about four times as many simulations.

In [ ]:
# HIGHLIGHTED ALGORITHM 7: Equity (Monte Carlo Rollout with Partial Fisher-Yates)

def equity(hole: List[int], board: List[int], num_opponents: int = 1, iters: int = 2000, rng: Optional[random.Random] = None) -> Dict[str, float]:
    "Algorithm 7 (Equity): Monte Carlo pot-share estimator with exact split-pot standard error."
    if rng is None:
        rng = random
    dead = set(hole) | set(board)
    stub = [c for c in FULL_DECK if c not in dead]
    K = len(stub)
    b_need = 5 - len(board)
    k_draw = b_need + 2 * num_opponents

    sim_hero = [hole[0], hole[1]] + list(board) + [0] * b_need
    sim_opp = [0, 0] + list(board) + [0] * b_need
    b_Len = len(board)

    w1 = 0.0
    w2 = 0.0
    for _ in range(iters):
        # Partial Fisher-Yates shuffle over the first k_draw slots of stub (no reset needed)
        for i in range(k_draw):
            j = i + int(rng.random() * (K - i))
            stub[i], stub[j] = stub[j], stub[i]

        for b in range(b_need):
            card_b = stub[b]
            sim_hero[2 + b_Len + b] = card_b
            sim_opp[2 + b_Len + b] = card_b

        v_hero = rank7fast(sim_hero)
        best_opp = -1
        ties = 0
        for p in range(num_opponents):
            sim_opp[0] = stub[b_need + 2 * p]
            sim_opp[1] = stub[b_need + 2 * p + 1]
            vp = rank7fast(sim_opp)
            if vp > best_opp:
                best_opp = vp
                ties = 1
            elif vp == best_opp:
                ties += 1

        if v_hero > best_opp:
            wt = 1.0
        elif v_hero == best_opp:
            wt = 1.0 / (ties + 1)
        else:
            wt = 0.0
        w1 += wt
        w2 += wt * wt

    q_hat = w1 / iters
    var_s = max(0.0, (w2 / iters) - q_hat * q_hat)
    se = math.sqrt(var_s / iters)
    return {"equity": q_hat, "se": se}




### 4.4 Range-conditioned equity (`equity_vs_range`)

A random opponent hand is a useful baseline, but a player who raises preflop is unlikely to hold every starting hand at the same frequency. Range-conditioned equity asks a conditional question:

\[
\mathbb{E}[X\mid \text{opponent hand passes a range filter}].
\]

Here the filter is `chen_score(opponent_cards) >= chen_floor`. Rejection sampling generates candidate opponent hole cards and discards hands below the threshold before evaluating the showdown. The accepted samples therefore follow the script's Chen-filtered range, subject to card removal.

This is still a heuristic range model: a Chen floor does not know the exact opponent's action frequencies, and a single cutoff cannot represent every player's mixed strategy. The function can return `None` if too few samples are accepted, so callers should use a fallback rather than interpreting `None` as zero equity.

In [ ]:
# HIGHLIGHTED ALGORITHM 8: EquityVsRange (Rejection-Sampled Range Rollout)

def equity_vs_range(hole: List[int], board: List[int], chen_floor: float, iters: int = 2000, rng: Optional[random.Random] = None) -> Optional[float]:
    "Algorithm 8 (EquityVsRange): Conditional equity given opponent Chen(H_opp) >= chen_floor."
    if rng is None:
        rng = random
    dead = set(hole) | set(board)
    stub = [c for c in FULL_DECK if c not in dead]
    K = len(stub)
    b_need = 5 - len(board)

    sim_hero = [hole[0], hole[1]] + list(board) + [0] * b_need
    sim_opp = [0, 0] + list(board) + [0] * b_need
    b_Len = len(board)

    w1 = 0.0
    accepted = 0
    attempts = 0
    max_attempts = iters * 14

    while accepted < iters and attempts < max_attempts:
        attempts += 1
        j0 = int(rng.random() * K)
        stub[0], stub[j0] = stub[j0], stub[0]
        j1 = 1 + int(rng.random() * (K - 1))
        stub[1], stub[j1] = stub[j1], stub[1]

        if CHEN_PAIR[stub[0]][stub[1]] < chen_floor:
            continue

        for b in range(b_need):
            idx = 2 + b
            jb = idx + int(rng.random() * (K - idx))
            stub[idx], stub[jb] = stub[jb], stub[idx]
            sim_hero[2 + b_Len + b] = stub[idx]
            sim_opp[2 + b_Len + b] = stub[idx]

        sim_opp[0], sim_opp[1] = stub[0], stub[1]
        v_hero = rank7fast(sim_hero)
        v_opp = rank7fast(sim_opp)
        if v_hero > v_opp:
            w1 += 1.0
        elif v_hero == v_opp:
            w1 += 0.5
        accepted += 1

    return (w1 / accepted) if accepted >= 30 else None




### 4.5 Exact river equity and the next-card scan

On a five-card board, with the hero's two cards known, 45 cards remain unseen. A heads-up opponent can have any unordered pair from those cards:

\[
\binom{45}{2}=990.
\]

Enumerating all 990 hands gives exact showdown equity for that known river board, with no Monte Carlo sampling error. On the turn, 46 river cards remain; for each candidate river, there are 990 opponent holdings after that river is fixed:

\[
46\times\binom{45}{2}=45,540 \text{ showdowns}.
\]

A next-card scan compares each card's equity against baseline equity. On the flop or in multiway situations, the script uses Monte Carlo estimates; a dynamic threshold based on the standard errors reduces the chance that simulation noise labels an ordinary card as helpful or harmful. “Helpful” and “scare” are relative equity changes in this model, not a guarantee that a card wins the hand.

In [ ]:
# HIGHLIGHTED ALGORITHM 9: ExactRiverEquity & NextCardScan

def exact_river_equity(hole: List[int], board5: List[int]) -> float:
    "Exact deterministic river equity over all C(45, 2) = 990 opponent hands."
    dead = set(hole) | set(board5)
    rem = [c for c in FULL_DECK if c not in dead]
    v_hero = rank7fast(hole + board5)
    opp_hand = [0, 0] + list(board5)
    wins = 0.0
    total = 0
    n_rem = len(rem)
    for i in range(n_rem):
        opp_hand[0] = rem[i]
        for j in range(i + 1, n_rem):
            opp_hand[1] = rem[j]
            v_opp = rank7fast(opp_hand)
            if v_hero > v_opp:
                wins += 1.0
            elif v_hero == v_opp:
                wins += 0.5
            total += 1
    return wins / total

def next_card_scan(hole: List[int], board: List[int], num_opponents: int = 1) -> Dict[str, object]:
    "Algorithm 9 (NextCardScan): Exact turn enumeration (45,540 deals, SE=0) or dynamic 2-sigma flop scan."
    dead = set(hole) | set(board)
    rem = [c for c in FULL_DECK if c not in dead]

    if len(board) == 4 and num_opponents == 1:
        card_eq = {}
        for c in rem:
            card_eq[c] = (exact_river_equity(hole, board + [c]), 0.0)
        base_eq = sum(v[0] for v in card_eq.values()) / len(rem)
        base_se = 0.0
    else:
        base_res = equity(hole, board, num_opponents, iters=600)
        base_eq, base_se = base_res["equity"], base_res["se"]
        card_eq = {}
        for c in rem:
            res_c = equity(hole, board + [c], num_opponents, iters=250)
            card_eq[c] = (res_c["equity"], res_c["se"])

    helpful_cards = []
    scare_cards = []
    for c, (eq_c, se_c) in card_eq.items():
        tau = max(0.03, 2.0 * math.sqrt(base_se**2 + se_c**2))
        delta = eq_c - base_eq
        if delta > tau:
            helpful_cards.append((c, delta))
        elif delta < -tau:
            scare_cards.append((c, delta))

    return {
        "base_equity": base_eq,
        "base_se": base_se,
        "helpful_count": len(helpful_cards),
        "scare_count": len(scare_cards),
        "helpful_cards": helpful_cards,
        "scare_cards": scare_cards,
    }

### Worked equity examples and timing demo

The next cell compares pocket Tens against a uniformly random hand, a looser Chen-filtered range, and a tighter Chen-filtered range. It then shows the exact heads-up turn scanner. Monte Carlo outputs can vary because the `equity_vs_range` examples use a random generator state that has been seeded and advanced by previous calls; timings vary by hardware and Python version.

In [ ]:
# Demonstrate Process III Benchmarks
seeded_rng = random.Random(42)
tt_hole = parse_cards("Ts Th")
kq5_flop = parse_cards("Ks Qd 5c")

eq_uniform = equity(tt_hole, kq5_flop, num_opponents=1, iters=4000, rng=seeded_rng)
eq_1raise = equity_vs_range(tt_hole, kq5_flop, chen_floor=6.1, iters=4000, rng=seeded_rng)
eq_3bet = equity_vs_range(tt_hole, kq5_flop, chen_floor=9.1, iters=4000, rng=seeded_rng)

print("Range-Conditioned Equity Benchmark: Pocket Tens (T♠ T♥) on K♠ Q♦ 5♣")
print(f"  1. vs Uniform Random Hand (Algorithm 7)         : {100*eq_uniform['equity']:.1f}% (SE = {100*eq_uniform['se']:.2f}%)")
print(f"  2. vs Single-Raise Range (Floor 6.1, Alg 8)     : {100*eq_1raise:.1f}%")
print(f"  3. vs 3-Bet Range (Floor 9.1, Alg 8)            : {100*eq_3bet:.1f}%\n")

# Exact Turn Enumeration Demo: Nut Flush Draw + Overcards (A♠ K♠) on Q♠ J♠ 4♥ 2♦
t0 = time.perf_counter()
scan_res = next_card_scan(parse_cards("As Ks"), parse_cards("Qs Js 4h 2d"), num_opponents=1)
dt_ms = (time.perf_counter() - t0) * 1000.0
print(f"Exact Turn Enumeration (Algorithm 9: A♠ K♠ on Q♠ J♠ 4♥ 2♦ across 46 x 990 = 45,540 deals in {dt_ms:.1f} ms):")
print(f"  Exact Turn Equity : {100*scan_res['base_equity']:.2f}% (Standard Error = {scan_res['base_se']:.4f})")
print(f"  Helpful River Outs: {scan_res['helpful_count']} cards -> {[card_str(c) for c, _ in scan_res['helpful_cards'][:12]]} ...")
print(f"  Scare River Cards : {scan_res['scare_count']} cards")

Range-Conditioned Equity Benchmark: Pocket Tens (T♠ T♥) on K♠ Q♦ 5♣
  1. vs Uniform Random Hand (Algorithm 7)         : 69.4% (SE = 0.73%)
  2. vs Single-Raise Range (Floor 6.1, Alg 8)     : 48.9%
  3. vs 3-Bet Range (Floor 9.1, Alg 8)            : 14.8%

Exact Turn Enumeration (Algorithm 9: A♠ K♠ on Q♠ J♠ 4♥ 2♦ across 46 x 990 = 45,540 deals in 197.1 ms):
  Exact Turn Equity : 63.10% (Standard Error = 0.0000)
  Helpful River Outs: 18 cards -> ['2♠', '3♠', '4♠', '5♠', '6♠', '7♠', '8♠', '9♠', 'T♠', 'T♥', 'T♦', 'T♣'] ...
  Scare River Cards : 28 cards


## 5. Processes IV and V — the betting state machine and pot settlement

### 5.1 State is the engine's memory of the hand

The engine keeps per-player data such as:

- `stack`: chips still available to wager;
- `bet`: chips put in during the current street;
- `committed`: chips contributed over the whole hand;
- `folded` and `all_in`: whether the player can continue;
- `has_acted`, `acted_at_bet`, and `min_raise_seen`: information used to decide whether action is due and whether raising has reopened.

The table tracks the current street, blind positions, current betting level, last full raise increment, board, and pot.

A central accounting goal is conservation of chips. Here the implementation stores `state.pot` and updates it when chips move; the intended consistency check is

\[
\sum_p p.stack + state.pot = \text{initial table chips},
\qquad state.pot=\sum_p p.committed
\]

while the hand is in progress (before settlement resets the pot). These equations should be checked after every state transition, not just at the end.

### Data structures and atomic chip transfer

The classes below separate a player from the table and a pot from its eligible winners. `transfer_chips` caps movement by the available stack and updates the stack, current-street bet, hand contribution, and stored pot together.

In [ ]:
@dataclass
class PlayerState:
    id: int
    name: str
    stack: int
    persona: str = "SOLID"
    is_human: bool = False
    hole: List[int] = field(default_factory=list)
    bet: int = 0
    committed: int = 0
    folded: bool = False
    all_in: bool = False
    has_acted: bool = False
    acted_at_bet: int = 0
    min_raise_seen: int = 0
    busted: bool = False
    score: int = -1
    best5: Tuple[int, ...] = ()

@dataclass
class PotSlice:
    amount: int
    eligible_ids: List[int]

@dataclass
class TableState:
    players: List[PlayerState]
    sb: int = 10
    bb: int = 20
    btn_seat: int = -1
    sb_seat: int = -1
    bb_seat: int = -1
    pot: int = 0
    current_bet: int = 0
    last_raise: int = 20
    raises: int = 0
    preflop_raises: int = 0
    aggressor_id: Optional[int] = None
    stage: str = "preflop"
    board: List[int] = field(default_factory=list)
    deck: List[int] = field(default_factory=list)
    acting_idx: int = -1


def transfer_chips(state: TableState, p: PlayerState, delta: int) -> int:
    "Atomic chip transfer conserving p.stack + state.pot (Equation 11)."
    actual = max(0, min(p.stack, delta))
    p.stack -= actual
    p.bet += actual
    p.committed += actual
    state.pot += actual
    if p.stack == 0 and p.committed > 0 and not p.folded:
        p.all_in = True
    return actual

### 5.2 Starting a hand: blinds and the button

`start_hand` resets per-hand state, deals two private cards to each funded player, posts the small and big blinds, then determines who acts first. The button and blinds are not merely display labels: they determine forced contributions and action order. Heads-up play is a special case because the button posts the small blind.

A short-stacked big blind may post fewer chips than the nominal big blind. The **preflop bring-in remains the full nominal big blind**; other players are not allowed to treat the short contribution as a smaller blind level. A dead-button tournament rule may leave the button or small blind on an empty chair, so position progression must be handled deliberately.

The Python reference's `start_hand` requires at least two funded players because it is also used to exercise heads-up and edge-case mechanics. A separate browser UI can impose a stricter minimum starting table size.

In [ ]:
# HIGHLIGHTED ALGORITHM 10: StartHand (Dead-Button Blind Advancement)

def next_funded_seat(players: List[PlayerState], from_seat: int) -> int:
    n = len(players)
    for k in range(1, n + 1):
        idx = (from_seat + k) % n
        if not players[idx].busted and players[idx].stack > 0:
            return idx
    return -1

def prev_funded_seat(players: List[PlayerState], from_seat: int) -> int:
    n = len(players)
    for k in range(1, n + 1):
        idx = (from_seat - k + n) % n
        if not players[idx].busted and players[idx].stack > 0:
            return idx
    return -1

def start_hand(state: TableState, custom_deck: Optional[List[int]] = None) -> None:
    "Algorithm 10 (StartHand): Dead-button blind advancement, short-BB bring-in, and deal."
    for p in state.players:
        p.busted = (p.stack <= 0)
        p.hole = []
        p.bet = 0
        p.committed = 0
        p.folded = p.busted
        p.all_in = False
        p.has_acted = False
        p.acted_at_bet = 0
        p.min_raise_seen = state.bb
        p.score = -1
        p.best5 = ()

    eligible = [p.id for p in state.players if not p.busted]
    assert len(eligible) >= 2, "Need at least 2 funded players"

    if state.bb_seat == -1:
        state.btn_seat = eligible[0]
        state.sb_seat = eligible[0] if len(eligible) == 2 else eligible[1]
        state.bb_seat = eligible[1] if len(eligible) == 2 else eligible[2 % len(eligible)]
    else:
        state.bb_seat = next_funded_seat(state.players, state.bb_seat)
        if len(eligible) == 2:
            state.sb_seat = prev_funded_seat(state.players, state.bb_seat)
            state.btn_seat = state.sb_seat  # Heads-up: Button posts Small Blind
        else:
            state.sb_seat = prev_funded_seat(state.players, state.bb_seat)
            state.btn_seat = prev_funded_seat(state.players, state.sb_seat)

    state.pot = 0
    state.board = []
    state.stage = "preflop"
    state.raises = 0
    state.preflop_raises = 0
    state.aggressor_id = None

    # Deal 2 hole cards to each funded player
    state.deck = list(custom_deck) if custom_deck is not None else shuffle_deck_in_place(list(FULL_DECK))
    for p in state.players:
        if not p.busted:
            p.hole = [state.deck.pop(), state.deck.pop()]

    # Post Small Blind and Big Blind
    transfer_chips(state, state.players[state.sb_seat], state.sb)
    transfer_chips(state, state.players[state.bb_seat], state.bb)

    # Full big-blind bring-in even if the BB player was short-stacked
    state.current_bet = max(state.bb, state.players[state.bb_seat].bet)
    state.last_raise = state.bb
    state.acting_idx = find_next_actor(state, state.bb_seat)



### 5.3 Finding the next player who owes an action

A betting round is not finished merely because everyone has acted once. A raise can force earlier players to act again. The next-actor predicate is essentially

\[
\neg p.has\_acted\quad\text{or}\quad p.bet < current\_bet.
\]

Folded and all-in players are skipped. If no eligible player satisfies the predicate, the round is closed. If only one player can still act and no further bet is possible, the engine should advance the street/runout rather than ask for meaningless actions.

In [ ]:
# HIGHLIGHTED ALGORITHM 11: FindNextActor (Turn-Order & Street Completion)

def find_next_actor(state: TableState, from_seat: int) -> int:
    "Algorithm 11 (FindNextActor): Scan clockwise for next player needing action, or -1."
    live = [p for p in state.players if not p.folded]
    if len(live) <= 1:
        return -1
    can_act = [p for p in live if not p.all_in]
    if len(can_act) == 0:
        return -1
    n = len(state.players)
    for k in range(1, n + 1):
        idx = (from_seat + k) % n
        p = state.players[idx]
        if p.folded or p.all_in:
            continue
        if (not p.has_acted) or (p.bet < state.current_bet):
            return idx
    return -1



### 5.4 Legal raises and cumulative short all-ins

A full raise must increase the current betting level by at least the previous full raise increment. If `current_bet` is the current level and `last_raise` is the last full increment, the standard target is at least

\[
\tau_{min}=current\_bet+last\_raise,
\]

subject to the player's maximum total contribution \(\tau_{max}=p.bet+p.stack\). A player can always make a legal all-in for less than this minimum; that short all-in may not reopen raising to a player who has already acted.

`can_reopen` tracks the betting level faced at the player's last action and the raise size that player last faced. Cumulative short all-ins can reopen action once the increase since that reference level reaches the relevant minimum threshold. Test this logic with scripted action sequences: it is subtle and should not be inferred from ordinary gameplay alone.

In [ ]:
# HIGHLIGHTED ALGORITHM 12: ApplyActionAndReopen (can_reopen, clamp_raise, apply)

def can_reopen(state: TableState, p: PlayerState) -> bool:
    "Algorithm 12a (CanReopen): True if player p is allowed to raise (including cumulative short all-ins)."
    to_call = max(0, state.current_bet - p.bet)
    if p.stack <= to_call:
        return False
    if not p.has_acted:
        return True
    return (state.current_bet - p.acted_at_bet) >= p.min_raise_seen

def clamp_raise(state: TableState, p: PlayerState, desired_to: float) -> int:
    "Algorithm 12b (ClampRaise): Round to table chip steps and clamp to [min_to, max_to]."
    max_to = p.bet + p.stack
    min_to = min(max_to, max(state.current_bet + state.last_raise, state.bb))
    step = 25 if state.bb >= 50 else (5 if state.bb >= 20 else 1)
    rounded = int(round(desired_to / step) * step)
    return min(max_to, max(min_to, rounded))

def apply_action(state: TableState, p: PlayerState, action: str, target_to: int = 0) -> None:
    "Algorithm 12c (ApplyAction): Execute Fold, Check, Call, or Raise and update street state."
    to_call = max(0, state.current_bet - p.bet)
    if action == "fold":
        p.folded = True
        p.has_acted = True
    elif action == "check":
        assert to_call == 0, "Cannot check when facing a bet"
        p.has_acted = True
        p.acted_at_bet = state.current_bet
        p.min_raise_seen = state.last_raise
    elif action == "call":
        transfer_chips(state, p, to_call)
        p.has_acted = True
        p.acted_at_bet = state.current_bet
        p.min_raise_seen = state.last_raise
    elif action == "raise":
        max_to = p.bet + p.stack
        min_to = min(max_to, max(state.current_bet + state.last_raise, state.bb))
        tau = min(max_to, max(min_to, target_to))
        delta = tau - p.bet
        raise_inc = tau - state.current_bet
        transfer_chips(state, p, delta)
        if raise_inc >= state.last_raise:
            # Full raise: updates last_raise and reopens action for all other active seats
            state.last_raise = raise_inc
            state.raises += 1
            if state.stage == "preflop":
                state.preflop_raises += 1
            state.aggressor_id = p.id
            for q in state.players:
                if q.id != p.id and not q.folded and not q.all_in:
                    q.has_acted = False
        state.current_bet = max(state.current_bet, p.bet)
        p.has_acted = True
        p.acted_at_bet = state.current_bet
        p.min_raise_seen = state.last_raise
    else:
        raise ValueError(f"Unknown action: {action}")

    state.acting_idx = find_next_actor(state, p.id)




### 5.5 Returning uncalled excess

If the highest contribution is greater than the second-highest contribution, the excess was not matched by any opponent and must be returned. If the two highest commitments are \(c_1\ge c_2\), the uncalled amount is

\[
\Delta=\max(0,c_1-c_2).
\]

Folded players must remain in contribution accounting: their chips stay in the pot, and in edge cases a folded player may be owed a refund for an uncalled portion. The current Python helper compares whole-hand `committed` totals and is intended for settlement; a production street-by-street engine should carefully distinguish current-street refunds from end-of-hand refunds.

In [ ]:
# HIGHLIGHTED ALGORITHM 13: RefundUncalled

def refund_uncalled(state: TableState) -> int:
    "Algorithm 13 (RefundUncalled): Return unmatched excess commitment to sole highest committer."
    committers = sorted([p for p in state.players if p.committed > 0], key=lambda x: x.committed, reverse=True)
    if not committers:
        return 0
    top = committers[0]
    second = committers[1].committed if len(committers) >= 2 else 0
    if top.committed > second:
        excess = top.committed - second
        top.committed -= excess
        top.stack += excess
        state.pot -= excess
        return excess
    return 0




### 5.6 Main pots and side pots by horizontal contribution bands

Players can contribute unequal amounts when they go all-in. A player can win only the portion of the pot that they were eligible to contest. The engine creates bands from sorted distinct commitment levels

\[
0=L_0<L_1<\cdots<L_k.
\]

For a player with total commitment \(c_p\), their contribution to band \((L_{j-1},L_j]\) is

\[
\gamma(p,L_{j-1},L_j)=\max\left(0,\min(c_p,L_j)-\min(c_p,L_{j-1})\right).
\]

The sum of contributions in a band is its amount. A player is eligible for a band only if they have not folded and their commitment reaches the band's top. Adjacent bands with the same eligibility set can be merged. Folded players contribute dead money but cannot win a pot.

In [ ]:
# HIGHLIGHTED ALGORITHM 14: ComputePots (Horizontal Commitment Slicing)
def compute_pots(state: TableState) -> List[PotSlice]:
    "Algorithm 14 (ComputePots): Horizontal side-pot slicing with orphaned-band backstop."
    levels = sorted({p.committed for p in state.players if p.committed > 0})
    pots: List[PotSlice] = []
    prev = 0
    last_eligible = [p.id for p in state.players if not p.folded]

    for lvl in levels:
        band_amount = 0
        eligible = []
        for p in state.players:
            contrib = max(0, min(p.committed, lvl) - min(p.committed, prev))
            band_amount += contrib
            if not p.folded and p.committed >= lvl:
                eligible.append(p.id)
        if eligible:
            last_eligible = eligible
        else:
            eligible = list(last_eligible)  # Orphaned-band backstop

        if band_amount > 0:
            if pots and pots[-1].eligible_ids == eligible:
                pots[-1].amount += band_amount
            else:
                pots.append(PotSlice(amount=band_amount, eligible_ids=eligible))
        prev = lvl

    if not pots and state.pot > 0:
        pots.append(PotSlice(amount=state.pot, eligible_ids=last_eligible))
    return pots




### 5.7 Showdown and odd chips

Each eligible player is ranked on the final board. For a pot amount \(A\) with \(m\) tied winners,

\[
share=\left\lfloor\frac{A}{m}\right\rfloor,\qquad remainder=A\bmod m.
\]

Everyone receives `share`; the remainder is distributed one chip at a time according to the table's odd-chip order (the script uses clockwise seat order beginning left of the button).

**Implementation caveat:** the current Python `compute_pots` uses the last non-empty eligibility set as a safety fallback if a band has no eligible claimant. That prevents an empty winner set, but it is not literally the same as refunding every contributor in an orphaned band. The fallback should be tested carefully before treating it as a full tournament-rules implementation.

In [ ]:
# HIGHLIGHTED ALGORITHM 15: AwardPots (Showdown Evaluation & Clockwise Odd Chips)
def award_pots(state: TableState) -> List[Dict[str, object]]:
    "Algorithm 15 (AwardPots): Evaluate showdown hands and award main/side pots + odd chips clockwise."
    refunded = refund_uncalled(state)
    live = [p for p in state.players if not p.folded]
    results = []

    if len(live) == 1:
        winner = live[0]
        amt = state.pot
        winner.stack += amt
        state.pot = 0
        results.append({"pot_index": 0, "amount": amt, "winners": [(winner.id, amt)], "refunded": refunded})
        return results

    for p in live:
        p.score, p.best5 = best7(p.hole + state.board)

    pots = compute_pots(state)
    n_seats = len(state.players)
    for idx, ps in enumerate(pots):
        elig_players = [state.players[i] for i in ps.eligible_ids]
        best_sc = max(p.score for p in elig_players)
        winners = [p for p in elig_players if p.score == best_sc]
        # Sort tied winners clockwise starting immediately left of the dealer button (btn_seat + 1)
        winners.sort(key=lambda w: (w.id - state.btn_seat - 1 + n_seats) % n_seats)

        share = ps.amount // len(winners)
        rem = ps.amount % len(winners)
        payouts = []
        for k, w in enumerate(winners):
            gain = share + (1 if k < rem else 0)
            w.stack += gain
            payouts.append((w.id, gain))
        results.append({"pot_index": idx, "amount": ps.amount, "winners": payouts, "refunded": refunded})

    state.pot = 0
    return results

### Worked side-pot slicing example

Player commitments are 100, 200, 350, and 350 chips. The level bands produce a 400-chip main pot eligible to Seats 0, 2, and 3, plus a 600-chip side pot eligible to Seats 2 and 3. Seat 1's 200-chip contribution remains in the main pot even though Seat 1 has folded.

In [ ]:
# Verify Example 7.1 (4-Player Multi-Way Side-Pot Decomposition) & Cumulative Short All-In Reopening
demo_players = [
    PlayerState(0, "Player A", stack=0, committed=100, all_in=True),
    PlayerState(1, "Player B", stack=200, committed=200, folded=True),
    PlayerState(2, "Player C", stack=0, committed=350, all_in=True),
    PlayerState(3, "Player D", stack=650, committed=350),
]
demo_state = TableState(players=demo_players, pot=1000)
sliced_pots = compute_pots(demo_state)
print("Worked 4-Player Multi-Way Side-Pot Decomposition (Algorithm 14):")
for i, sl in enumerate(sliced_pots):
    label = "Main Pot" if i == 0 else f"Side Pot {i}"
    print(f"  {label:<10}: Amount = {sl.amount:>4d} chips | Eligible Seat IDs = {sl.eligible_ids}")
assert [(p.amount, p.eligible_ids) for p in sliced_pots] == [(400, [0, 2, 3]), (600, [2, 3])]

Worked 4-Player Multi-Way Side-Pot Decomposition (Algorithm 14):
  Main Pot  : Amount =  400 chips | Eligible Seat IDs = [0, 2, 3]
  Side Pot 1: Amount =  600 chips | Eligible Seat IDs = [2, 3]


## 6. Process VI — AI policy, game theory, and opponent adaptation

### 6.1 Player personas and difficulty

The code uses a few named personas with parameters for tightness, aggression, and bluffing. They are adjustable policy parameters, not learned universal poker constants. Difficulty interpolates a persona's parameters toward the `SOLID` persona:

\[
\theta_{eff}=(1-\lambda)\theta_{persona}+\lambda\theta_{solid}.
\]

This is a transparent way to make the policies more alike at higher difficulty without changing the rules engine.

In [ ]:
# PERSONA ARCHETYPES & DIFFICULTY INTERPOLATION
PERSONAS = {
    "TAG":   {"tight": 0.06, "aggr": 0.62, "bluff": 0.14},
    "LAG":   {"tight": 0.02, "aggr": 0.72, "bluff": 0.30},
    "CALL":  {"tight": 0.02, "aggr": 0.38, "bluff": 0.12},
    "NIT":   {"tight": 0.10, "aggr": 0.45, "bluff": 0.08},
    "SOLID": {"tight": 0.04, "aggr": 0.58, "bluff": 0.14},
}
DIFF_PULL = {"casual": 0.00, "solid": 0.35, "sharp": 0.70}

def effective_persona(code: str, difficulty: str = "solid") -> Dict[str, float]:
    base = PERSONAS.get(code, PERSONAS["SOLID"])
    solid = PERSONAS["SOLID"]
    lam = DIFF_PULL.get(difficulty, 0.35)
    return {
        "tight": (1.0 - lam) * base["tight"] + lam * solid["tight"],
        "aggr":  (1.0 - lam) * base["aggr"]  + lam * solid["aggr"],
        "bluff": (1.0 - lam) * base["bluff"] + lam * solid["bluff"],
    }

### 6.2 169 starting-hand classes and the push/fold demonstration

Although there are 1,326 physical two-card combinations, many have the same preflop structure: pocket pairs, suited non-pairs, and offsuit non-pairs. These compress into 169 strategic classes. In a 13-by-13 chart, pairs occupy the diagonal, suited hands one triangle, and offsuit hands the other.

**Important source limitation:** `solve_push_fold_cfr_demo` is a small teaching demonstration. It builds an approximate pairwise equity matrix from Chen-score differences using a logistic/Bradley–Terry-style curve, then runs a simplified regret-matching update. It does **not** enumerate exact card-vs-card equity and is **not** a complete solver for the full 169-by-19 Nash push/fold game. Therefore its output is an illustration of iterative strategy adjustment, not a validated Nash equilibrium table.

In [ ]:
# HIGHLIGHTED ALGORITHM 17: PushFold Lookup & Regret-Matching CFR Solver
def hand_to_169_index(c1: int, c2: int) -> int:
    "Map any 2 hole cards (c1, c2) into its canonical 13x13 grid index in {0..168}."
    r1, r2 = c1 >> 2, c2 >> 2
    s1, s2 = c1 & 3, c2 & 3
    hi, lo = 12 - max(r1, r2), 12 - min(r1, r2)
    if r1 == r2 or s1 == s2:
        row, col = hi, lo  # Upper triangle (suited) or diagonal (pair)
    else:
        row, col = lo, hi  # Lower triangle (offsuit)
    return row * 13 + col

def solve_push_fold_cfr_demo(stack_bb: float = 10.0, iterations: int = 800) -> Tuple[float, float]:
    '''
    Demonstrate the Counterfactual Regret Minimization (Regret Matching) process
    on a simplified 169-class Heads-Up Push/Fold game at stack depth B = stack_bb.
    '''
    # Build canonical representative hand and combo weight (6 pair, 4 suited, 12 offsuit) for each of 169 cells
    weights = [0.0] * 169
    chen_169 = [0.0] * 169
    for row in range(13):
        for col in range(13):
            idx = row * 13 + col
            r_hi, r_lo = 12 - min(row, col), 12 - max(row, col)
            if row == col:
                weights[idx] = 6.0
                chen_169[idx] = chen_score(encode_card(r_hi, 0), encode_card(r_lo, 1))
            elif row < col:
                weights[idx] = 4.0
                chen_169[idx] = chen_score(encode_card(r_hi, 0), encode_card(r_lo, 0))
            else:
                weights[idx] = 12.0
                chen_169[idx] = chen_score(encode_card(r_hi, 0), encode_card(r_lo, 1))

    # Smooth Bradley-Terry equity approximation from Chen scores for fast self-contained notebook execution
    eq_matrix = [[0.5] * 169 for _ in range(169)]
    for i in range(169):
        for j in range(i + 1, 169):
            p_ij = 1.0 / (1.0 + math.exp(-0.18 * (chen_169[i] - chen_169[j])))
            eq_matrix[i][j] = p_ij
            eq_matrix[j][i] = 1.0 - p_ij

    regret_sb = [0.0] * 169  # Cumulative regret for Pushing vs Folding (-0.5 bb)
    regret_bb = [0.0] * 169  # Cumulative regret for Calling vs Folding (-1.0 bb)
    sum_push = [0.0] * 169
    sum_call = [0.0] * 169

    for t in range(1, iterations + 1):
        # Regret Matching: convert positive cumulative regrets into current strategy probabilities
        s_push = [1.0 if regret_sb[i] > 0 else 0.0 for i in range(169)]
        c_call = [1.0 if regret_bb[j] > 0 else 0.0 for j in range(169)]
        if t == 1:
            s_push = [0.5] * 169
            c_call = [0.5] * 169

        for i in range(169):
            sum_push[i] += s_push[i]
            sum_call[i] += c_call[i]

        # Update SB regrets against BB's current calling strategy
        for i in range(169):
            ev_push = 0.0
            w_sum = 0.0
            for j in range(169):
                w = weights[j]
                w_sum += w
                ev_push += w * ((1.0 - c_call[j]) * 1.0 + c_call[j] * stack_bb * (2.0 * eq_matrix[i][j] - 1.0))
            ev_push /= w_sum
            regret_sb[i] = max(0.0, regret_sb[i] + (ev_push - (-0.5)))

        # Update BB regrets against SB's current pushing strategy
        for j in range(169):
            ev_call = 0.0
            w_sum = 0.0
            for i in range(169):
                w = weights[i] * s_push[i]
                w_sum += w
                ev_call += w * ((1.0 - eq_matrix[i][j]) * (2.0 * stack_bb) - (stack_bb - 1.0))
            if w_sum > 0:
                ev_call /= w_sum
                regret_bb[j] = max(0.0, regret_bb[j] + (ev_call - (-1.0)))

    avg_push = [sum_push[i] / iterations for i in range(169)]
    avg_call = [sum_call[j] / iterations for j in range(169)]
    combo_push_pct = sum(weights[i] * avg_push[i] for i in range(169)) / 1326.0
    combo_call_pct = sum(weights[j] * avg_call[j] for j in range(169)) / 1326.0
    return combo_push_pct, combo_call_pct

### 6.3 Bayesian opponent reads

The engine tracks observed tendencies such as voluntarily entering pots, preflop raises, and folds to bets. A Beta distribution is a convenient model for an unknown probability \(p\) between 0 and 1. With prior \(p\sim\mathrm{Beta}(a_0,b_0)\), after \(k\) successes in \(n\) observations the posterior is

\[
p\mid data\sim\mathrm{Beta}(a_0+k,b_0+n-k).
\]

Its mean is \((a_0+k)/(a_0+b_0+n)\); its variance is \(ab/[(a+b)^2(a+b+1)]\). Priors keep estimates from jumping to 0% or 100% after only a few observations.

The code applies a two-standard-deviation-style gate before emitting a nonzero exploitative signal. This is a practical heuristic gate—not a formal frequentist significance test, and not a full posterior probability that the opponent differs from the prior.

In [ ]:
# HIGHLIGHTED ALGORITHM 19: BayesianRead (Beta-Binomial Conjugate Profiling)

PRIORS = {
    "vpip":        (5.0, 7.0),  # Prior mean = 41.7%
    "pfr":         (3.0, 9.0),  # Prior mean = 25.0%
    "foldToRaise": (5.0, 5.0),  # Prior mean = 50.0%
    "foldToBet":   (6.0, 6.0),  # Prior mean = 50.0%
    "aggr":        (4.0, 6.0),  # Prior mean = 40.0%
}

def bayesian_stat(key: str, k_succ: int, n_obs: int) -> Dict[str, float]:
    "Algorithm 19 (BayesianRead): Beta-Binomial posterior mean, std, and 2-sigma gated signal z."
    a0, b0 = PRIORS[key]
    a = a0 + k_succ
    b = b0 + (n_obs - k_succ)
    n = a + b
    mu0 = a0 / (a0 + b0)
    mu = a / n
    sigma = math.sqrt((a * b) / (n * n * (n + 1.0)))
    delta = mu - mu0
    z = max(-1.0, min(1.0, delta / (4.0 * sigma))) if abs(delta) > 2.0 * sigma else 0.0
    return {"mean": mu, "prior_mean": mu0, "sigma": sigma, "z": z, "n_obs": n_obs}

### 6.4 Postflop decision policy: equity against a range

The postflop policy estimates equity, then compares an adjusted equity estimate against a calling threshold influenced by pot odds, player parameters, and repeated raises. The opponent's preflop actions are used to select a Chen-score floor: no preflop raise is treated as a wider range; one raise as narrower; multiple raises as narrower still.

When a conditioned range is available, the code applies a smaller square-root pressure discount to avoid counting the opponent's range strength twice:

\[
\hat e=e-\min(c_{max},\gamma\sqrt{pressure})-0.025(m-1),
\qquad pressure=\frac{toCall}{\max(bb,pot)}.
\]

This is a rule-based policy. The Chen floor and discount are heuristics, so their thresholds should be calibrated by simulated results and decision-quality evaluation rather than described as equilibrium values.

In [ ]:
# HIGHLIGHTED ALGORITHM 18: PostflopDecision (Range Floor & Pressure Discount)
def postflop_chen_floor(state: TableState, opp_reads: Dict[str, Dict[str, float]]) -> float:
    "Compute action-conditioned opponent Chen floor c_floor in [2, 10] (Equation 17)."
    if state.preflop_raises >= 2:
        floor = 8.5
    elif state.preflop_raises == 1:
        floor = 5.5
    else:
        floor = 2.5

    if state.preflop_raises >= 1 and "pfr" in opp_reads:
        r = opp_reads["pfr"]
        if r["n_obs"] >= 6:
            floor += max(-1.2, min(1.2, (0.22 - r["mean"]) * 5.0))
    elif "vpip" in opp_reads:
        r = opp_reads["vpip"]
        if r["n_obs"] >= 6:
            floor += max(-1.2, min(1.2, (0.42 - r["mean"]) * 4.0))

    if state.raises >= 2:
        floor += 0.8
    return max(2.0, min(10.0, floor))

def ai_postflop_decision(state: TableState, p: PlayerState, opp_reads: Optional[Dict[str, Dict[str, float]]] = None) -> Tuple[str, int, Dict[str, float]]:
    "Algorithm 18 (PostflopDecision): Action-conditioned range equity, pressure discount, and action choice."
    opp_reads = opp_reads or {}
    to_call = max(0, state.current_bet - p.bet)
    pot_odds = to_call / (state.pot + to_call) if to_call > 0 else 0.0
    live_opps = [q for q in state.players if not q.folded and q.id != p.id]
    m = max(1, len(live_opps))
    per = effective_persona(p.persona, "solid")

    used_range = False
    eq_val = None
    if m == 1:
        c_floor = postflop_chen_floor(state, opp_reads) + (0.6 if to_call > 0 else 0.0)
        eq_val = equity_vs_range(p.hole, state.board, min(10.0, c_floor), iters=900)
        if eq_val is not None:
            used_range = True
    if eq_val is None:
        eq_val = equity(p.hole, state.board, num_opponents=m, iters=900)["equity"]

    # Square-root bet-pressure discount (halved when used_range=True to avoid double discounting)
    pressure = to_call / max(state.bb, state.pot)
    gamma_p, cap_p = (0.06, 0.10) if used_range else (0.15, 0.20)
    adj_eq = eq_val - min(cap_p, gamma_p * math.sqrt(pressure)) - 0.025 * (m - 1)

    can_r = can_reopen(state, p)
    if to_call == 0:
        if eq_val > 0.62 and can_r:
            target = clamp_raise(state, p, p.bet + 0.66 * state.pot)
            return "raise", target, {"raw_eq": eq_val, "adj_eq": adj_eq, "used_range": used_range}
        return "check", 0, {"raw_eq": eq_val, "adj_eq": adj_eq, "used_range": used_range}

    need = pot_odds + per["tight"] + (0.07 if state.raises >= 3 else 0.0) + (0.05 if to_call >= p.stack else 0.0)
    if eq_val > 0.74 and can_r:
        target = clamp_raise(state, p, state.current_bet + 0.75 * (state.pot + to_call))
        return "raise", target, {"raw_eq": eq_val, "adj_eq": adj_eq, "need": need}
    if adj_eq >= need:
        return "call", 0, {"raw_eq": eq_val, "adj_eq": adj_eq, "need": need}
    return "fold", 0, {"raw_eq": eq_val, "adj_eq": adj_eq, "need": need}

### 6.5 Post-hand self-review: an important implementation caveat

The intended concept is to evaluate a past decision using what was knowable at that moment, avoiding the classic mistake of judging a decision only by whether the river happened to help. Unrevealed folded hands should not be peeked at.

**Code-vs-description note:** `review_hand_trace` builds `known_opps`, but the current call to `equity(...)` passes only the number of opponents, not the known opponent hole cards themselves. Thus this implementation re-samples all opponents as random hands; it does not actually condition on the exact shown hands. Treat it as a rough demonstration of regret buckets until that gap is corrected.

In [ ]:
# HIGHLIGHTED ALGORITHM 20: ReviewHand (Hindsight-Free Self-Review)
def review_hand_trace(trace: List[Dict[str, object]], shown_hands: Dict[int, List[int]], learn_store: Dict[str, List[float]], bb: int = 20) -> None:
    "Algorithm 20 (ReviewHand): Update EV regret buckets using only showdown-revealed hands (mucked stay hidden)."
    for d in trace:
        if d["to_call"] <= 0:
            continue
        known_opps = [shown_hands[oid] for oid in d["opp_ids"] if oid in shown_hands]
        unknown_mucked = sum(1 for oid in d["opp_ids"] if oid not in shown_hands)
        # Compute counterfactual equity at decision time against revealed hands + random mucked hands
        q_true = equity(d["hole"], d["board"], num_opponents=len(known_opps) + unknown_mucked, iters=400)["equity"]
        ev_fold = 0.0
        ev_call = q_true * d["pot"] - (1.0 - q_true) * d["to_call"]
        base_ev = ev_fold if d["action"] == "fold" else ev_call
        bucket = learn_store.setdefault(d["key"], [0.0, 0.0, 0])
        bucket[0] += (ev_fold - base_ev) / bb
        bucket[1] += (ev_call - base_ev) / bb
        bucket[2] += 1

### Demonstrations: toy push/fold iterations and Bayesian updates

The strategy numbers below are outputs of the simplified demonstration described above. They should not be copied into a real strategy chart without stronger validation.

In [ ]:
# Demonstrate CFR Push/Fold Convergence & Bayesian Two-Sigma Gating
print("CFR / Regret-Matching Push/Fold Equilibrium Convergence across Stack Depths:")
for depth in [15.0, 10.0, 5.0]:
    p_shove, p_call = solve_push_fold_cfr_demo(stack_bb=depth, iterations=600)
    print(f"  Stack Depth {depth:>4.1f} bb -> SB Equilibrium Shove: {100*p_shove:>5.1f}% | BB Equilibrium Call: {100*p_call:>5.1f}%")

print("\nBayesian Opponent Profiling (Algorithm 19: Beta-Binomial 2-Sigma Gate on foldToBet):")
for k_folds, n_trials in [(2, 2), (5, 6), (12, 14), (20, 24)]:
    st = bayesian_stat("foldToBet", k_folds, n_trials)
    status = "GATED (z = 0, noise filtered)" if st["z"] == 0 else f"ACTIVE (z = {st['z']:+.3f})"
    print(f"  Observed {k_folds:>2d}/{n_trials:<2d} folds -> Posterior Mean = {100*st['mean']:5.1f}% ± {100*st['sigma']:4.1f}% | {status}")

CFR / Regret-Matching Push/Fold Equilibrium Convergence across Stack Depths:
  Stack Depth 15.0 bb -> SB Equilibrium Shove:  43.8% | BB Equilibrium Call:  37.5%
  Stack Depth 10.0 bb -> SB Equilibrium Shove:  49.7% | BB Equilibrium Call:  49.8%
  Stack Depth  5.0 bb -> SB Equilibrium Shove:  63.9% | BB Equilibrium Call:  86.3%

Bayesian Opponent Profiling (Algorithm 19: Beta-Binomial 2-Sigma Gate on foldToBet):
  Observed  2/2  folds -> Posterior Mean =  57.1% ± 12.8% | GATED (z = 0, noise filtered)
  Observed  5/6  folds -> Posterior Mean =  61.1% ± 11.2% | GATED (z = 0, noise filtered)
  Observed 12/14 folds -> Posterior Mean =  69.2% ±  8.9% | ACTIVE (z = +0.541)
  Observed 20/24 folds -> Posterior Mean =  72.2% ±  7.4% | ACTIVE (z = +0.754)


## 7. Verification and a complete worked hand

### 7.1 Core invariants

An invariant is a property that should remain true after every valid state transition. The reference documentation lists checks such as:

1. **Chip conservation:** stacks plus committed chips equal the initial table total.
2. **Settlement conservation:** after awards, all chips are back in player stacks.
3. **Legal betting level:** a live, non-all-in player cannot be above the current matching level except during a controlled transition.
4. **Card uniqueness:** no physical card appears twice among live hole cards and the board.
5. **Pot eligibility:** each pot has a valid eligible winner or its contributions are returned.
6. **Raise legality:** raises respect the minimum increment unless the raise is an all-in.
7. **Nonnegative stacks:** no player has a negative chip balance.

The helper `assert_engine_invariants` in the supplied script directly checks chip conservation, the consistency of `sum(committed)` with `pot`, and nonnegative stack/bet constraints. It does not by itself implement every one of the seven statements, so the final printed phrase should be read as “the walkthrough's assertions passed,” not as a formal proof of all seven invariants.

### 7.2 Walkthrough setup

We replay a three-player example with small blind 10 and big blind 20. Seat 0 holds J♠ J♥, Seat 1 holds Q♠ T♠, and Seat 2 holds A♥ K♦. Seat 2 raises preflop to 55 and the other players call. The flop is Q♥ 8♠ 4♣. Seat 0 checks, Seat 1 bets 100, and Seat 2 shoves to 185—an increase of 85, below the previous full raise increment of 100. Seat 0 has not yet acted against that wager level and retains raising rights; Seat 1 already bet 100 and has not faced a full 100-chip raise increment, so action has not reopened to that player under this model.

The revised notebook creates a deterministic deck order for this walkthrough so the hole cards and board do not duplicate cards in the remaining deck. The code then settles the 535-chip pot and verifies chip conservation.

In [ ]:
def assert_engine_invariants(state: TableState, initial_total_chips: int) -> None:
    "Verify Formal Engine Invariants I1 (Chip Conservation) and I2 (Stack/Commitment Bounds)."
    current_stacks = sum(p.stack for p in state.players)
    assert current_stacks + state.pot == initial_total_chips, (
        f"Invariant I1 violated: stacks ({current_stacks}) + pot ({state.pot}) != {initial_total_chips}"
    )
    if state.pot > 0:
        assert sum(p.committed for p in state.players) == state.pot, "Invariant I1b violated: sum(committed) != pot"
    for p in state.players:
        assert p.stack >= 0 and p.bet >= 0 and p.committed >= p.bet, "Invariant I2 violated: negative stack/bet"


# Execute the Exact 3-Player Numerical Walkthrough from Section 10.3 of the Documentation
walkthrough_players = [
    PlayerState(0, "Seat 0 (TAG)",   stack=1000, persona="TAG"),
    PlayerState(1, "Seat 1 (Human)", stack=1000, persona="SOLID", is_human=True),
    PlayerState(2, "Seat 2 (LAG)",   stack=240,  persona="LAG"),
]
# Prior hand had bb_seat = 0, so StartHand advances BB clockwise to Seat 1, SB to Seat 0, BTN to Seat 2
w_state = TableState(players=walkthrough_players, sb=10, bb=20, bb_seat=0)
initial_M0 = sum(p.stack for p in w_state.players)

# Arrange the deck tail so pop() deals the hole cards first, followed by the board.
desired_sequence = parse_cards("Js Jh Qs Ts Ah Kd Qh 8s 4c 2d 7c")
custom_deal = [c for c in FULL_DECK if c not in desired_sequence] + list(reversed(desired_sequence))
w_state.bb_seat = 0
start_hand(w_state, custom_deck=custom_deal)
assert [p.hole for p in w_state.players] == [
    parse_cards("Js Jh"), parse_cards("Qs Ts"), parse_cards("Ah Kd")
]
assert_engine_invariants(w_state, initial_M0)

print("=== STEP 1: StartHand & Dead-Button Blind Advancement (Algorithm 10) ===")
print(f"  Positions : BTN = Seat {w_state.btn_seat}, SB = Seat {w_state.sb_seat} (10), BB = Seat {w_state.bb_seat} (20)")
print(f"  Hole Cards: Seat 0 = {hand_str(w_state.players[0].hole)} (Chen={chen_score(*w_state.players[0].hole):.1f}), "
      f"Seat 1 = {hand_str(w_state.players[1].hole)} (Chen={chen_score(*w_state.players[1].hole):.1f}), "
      f"Seat 2 = {hand_str(w_state.players[2].hole)} (Chen={chen_score(*w_state.players[2].hole):.1f})")
print(f"  First Actor Preflop: Seat {w_state.acting_idx} | Pot = {w_state.pot}")

print("\n=== STEP 2: Preflop Betting Round (Algorithms 11 & 12) ===")
apply_action(w_state, w_state.players[2], "raise", target_to=55)  # Seat 2 (BTN) raises to 55
apply_action(w_state, w_state.players[0], "call")                 # Seat 0 (SB) calls 45 more (to 55)
apply_action(w_state, w_state.players[1], "call")                 # Seat 1 (BB) calls 35 more (to 55)
assert w_state.acting_idx == -1, "Preflop betting round should be complete (acting_idx == -1)"
assert_engine_invariants(w_state, initial_M0)
print(f"  Preflop Complete: Pot = {w_state.pot}, All active players committed = {[p.committed for p in w_state.players]}")

print("\n=== STEP 3: Flop Q♥ 8♠ 4♣ & Short All-In Reopening Rule (Algorithm 12) ===")
w_state.stage = "flop"
w_state.board = [w_state.deck.pop() for _ in range(3)]
assert w_state.board == parse_cards("Qh 8s 4c")
for p in w_state.players:
    p.bet = 0
    p.has_acted = False
    p.acted_at_bet = 0
    p.min_raise_seen = w_state.bb
w_state.current_bet = 0
w_state.last_raise = w_state.bb
w_state.raises = 0
w_state.acting_idx = find_next_actor(w_state, w_state.btn_seat)

apply_action(w_state, w_state.players[0], "check")                 # Seat 0 checks
apply_action(w_state, w_state.players[1], "raise", target_to=100)  # Seat 1 bets 100 (last_raise = 100)
apply_action(w_state, w_state.players[2], "raise", target_to=185)  # Seat 2 shoves remaining 185 (short raise +85 < 100!)

print(f"  After Seat 2 short all-in to 185 (+85 < last_raise=100):")
print(f"    Can Seat 0 (who checked at 0) reopen raising? {can_reopen(w_state, w_state.players[0])} (185 - 0 >= 20)")
print(f"    Can Seat 1 (who bet 100) reopen raising?      {can_reopen(w_state, w_state.players[1])} (185 - 100 = 85 < 100)")

apply_action(w_state, w_state.players[0], "fold")                  # Seat 0 folds J♠ J♥ (leaves 55 dead chips)
assert not can_reopen(w_state, w_state.players[1])
apply_action(w_state, w_state.players[1], "call")                  # Seat 1 calls the 85 short all-in
assert w_state.acting_idx == -1
assert_engine_invariants(w_state, initial_M0)
print(f"  Flop Complete: Pot = {w_state.pot} (Seat 0 folded 55, Seat 1 & Seat 2 committed 240 each)")

print("\n=== STEP 4: Turn 2♦ & River 7♣ Runout + Showdown Settlement (Algorithms 13, 14, 15) ===")
w_state.board.extend([w_state.deck.pop(), w_state.deck.pop()])
assert w_state.board == parse_cards("Qh 8s 4c 2d 7c")
settlement = award_pots(w_state)
assert_engine_invariants(w_state, initial_M0)

for res in settlement:
    print(f"  Pot {res['pot_index']} ({res['amount']} chips) awarded to: {res['winners']}")
for p in w_state.players:
    hand_desc = format_hand_name(p.score) if p.score >= 0 else "Folded"
    print(f"  {p.name:<15}: Final Stack = {p.stack:>4d} | Showdown = {hand_desc} ({p.score})")
print(f"\nWalkthrough assertions passed: Total Table Chips = {sum(p.stack for p in w_state.players)} == M_0 ({initial_M0})!")

=== STEP 1: StartHand & Dead-Button Blind Advancement (Algorithm 10) ===
  Positions : BTN = Seat 2, SB = Seat 0 (10), BB = Seat 1 (20)
  Hole Cards: Seat 0 = J♠ J♥ (Chen=12.0), Seat 1 = Q♠ T♠ (Chen=8.0), Seat 2 = A♥ K♦ (Chen=10.0)
  First Actor Preflop: Seat 2 | Pot = 30

=== STEP 2: Preflop Betting Round (Algorithms 11 & 12) ===
  Preflop Complete: Pot = 165, All active players committed = [55, 55, 55]

=== STEP 3: Flop Q♥ 8♠ 4♣ & Short All-In Reopening Rule (Algorithm 12) ===
  After Seat 2 short all-in to 185 (+85 < last_raise=100):
    Can Seat 0 (who checked at 0) reopen raising? True (185 - 0 >= 20)
    Can Seat 1 (who bet 100) reopen raising?      False (185 - 100 = 85 < 100)
  Flop Complete: Pot = 535 (Seat 0 folded 55, Seat 1 & Seat 2 committed 240 each)

=== STEP 4: Turn 2♦ & River 7♣ Runout + Showdown Settlement (Algorithms 13, 14, 15) ===
  Pot 0 (535 chips) awarded to: [(1, 535)]
  Seat 0 (TAG)   : Final Stack =  945 | Showdown = Folded (-1)
  Seat 1 (Human) : Final Stack

## 8. Final interpretation and development checklist

### What this implementation demonstrates well

- compact card IDs and unbiased Fisher-Yates shuffling;
- a readable exact evaluator plus a faster bitmask evaluator and randomized differential check;
- Monte Carlo pot-share estimates with a reported standard error;
- range-conditioned sampling and exact heads-up turn-to-river enumeration;
- explicit betting state and chip transfers;
- main/side-pot decomposition and odd-chip distribution;
- illustrative strategic heuristics, Bayesian updating, and post-hand review concepts.

### What should not be overclaimed

- A random-hand equity estimate is not a read against a betting opponent.
- A Chen threshold is a proxy for a range, not an exact opponent model.
- The included CFR code is a simplified teaching demo, not a full poker-game equilibrium solver.
- The self-review code does not currently use the actual revealed hole-card values when estimating equity.
- The invariant helper directly verifies only a subset of the formal invariant list.
- A successful randomized differential test gives strong evidence but is not exhaustive proof over every possible deal.

### Suggested next tests

1. Add validation for duplicated cards and invalid hand/board lengths at public function boundaries.
2. Add exhaustive five-card category tests and a larger seeded `rank7fast` versus `best7` test suite.
3. Test short blinds, dead buttons, full raises, cumulative short all-ins, folded contributions, and odd-chip ties with fixed scenarios.
4. Replace the toy push/fold model with a solver whose game tree, utilities, convergence criteria, and exploitability metric are explicitly defined.
5. Implement equity against specified known opponent cards for review logic, preserving card-removal constraints.
6. Use reproducible benchmarks and report hardware, Python version, trial count, and timing distribution.

### Source document

Prepared from the supplied `no_limit_holdem_engine_algorithms.py` and the attached *Algorithms and Processes of a No-Limit Hold'em Engine: A First-Principles Reference* PDF (revision dated 20 October 2026). The notebook follows its process structure while annotating specific cases where the source implementation is narrower than the prose description.

## Appendix A — optional export utility

The original script includes a utility to convert a percent-formatted `.py` file into a Jupyter notebook. In a notebook environment `__file__` is not normally defined, so this adapted version requires the source path explicitly.

In [ ]:
import json as _json

def export_this_file_to_ipynb(source_py_path: str, output_ipynb_path: str = "no_limit_holdem_engine_algorithms.ipynb") -> str:
    "Convert this # %% percent-formatted Python notebook into a standard v4 .ipynb file."
    with open(source_py_path, "r", encoding="utf-8") as f:
        raw_lines = f.read().splitlines()

    nb_cells = []
    cur_type = None
    cur_lines = []

    def flush_cell():
        nonlocal cur_type, cur_lines
        if cur_type is None:
            return
        while cur_lines and not cur_lines[-1].strip():
            cur_lines.pop()
        formatted = [l + "\n" for l in cur_lines]
        if formatted:
            formatted[-1] = formatted[-1].rstrip("\n")
        if cur_type == "markdown":
            nb_cells.append({"cell_type": "markdown", "metadata": {}, "source": formatted})
        else:
            nb_cells.append({"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": formatted})
        cur_type = None
        cur_lines = []

    for line in raw_lines:
        if line.startswith("# %% [markdown]"):
            flush_cell()
            cur_type = "markdown"
        elif line.startswith("# %%"):
            flush_cell()
            cur_type = "code"
        elif cur_type == "markdown":
            cur_lines.append(line[2:] if line.startswith("# ") else (line[1:] if line.startswith("#") else line))
        elif cur_type == "code":
            cur_lines.append(line)
    flush_cell()

    nb_doc = {
        "cells": nb_cells,
        "metadata": {
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3.10"}
        },
        "nbformat": 4,
        "nbformat_minor": 5
    }
    with open(output_ipynb_path, "w", encoding="utf-8") as out_f:
        _json.dump(nb_doc, out_f, indent=2, ensure_ascii=False)
    return output_ipynb_path